# Week 0 · The implicit world model

A transformer learns to predict legal Othello moves from move sequences alone. It never sees a board. The question: has it built one anyway?

Three steps, under an hour on a free Colab T4 (Runtime → Change runtime type → T4 GPU):

1. **Train** Othello-GPT on synthetic random games.
2. **Probe**: fit a linear map from its activations to the board. Compare with the same probe on an untrained network, the control that keeps the result honest.
3. **Intervene**: edit the activations to flip one tile, and check that the model's move predictions follow the *edited* board.

The plan suggests cloning `likenneth/othello_world`. That repo's checkpoints sit on Google Drive links that break without warning, so this notebook trains its own small model instead, from a self-contained engine in `wmlab/othello`. The science is the same; the dependency is gone.

**Deliverable:** probe accuracy by layer, and one successful causal intervention.

In [ ]:
# --- Setup: runs on Colab or locally (from the notebooks/ folder) -----------
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/world-models-lab.git"   # <- edit once
USE_DRIVE = True        # keep data and checkpoints on Google Drive across disconnects

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("WM_SMOKE") == "1"      # tiny sizes, used only to test the notebook
if IN_COLAB:
    if not os.path.exists("/content/world-models-lab"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/world-models-lab"], check=True)
    os.chdir("/content/world-models-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "swig"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/world-models-lab"
    else:
        WORK = "/content/work"
else:
    sys.path.insert(0, os.path.abspath(".."))
    WORK = os.path.abspath(os.environ.get("WM_WORK", "../work"))
os.makedirs(WORK, exist_ok=True)

import numpy as np, torch, matplotlib.pyplot as plt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| work dir:", WORK, "| smoke test:", SMOKE)

## 1 · Synthetic games
Uniformly random legal games. Random play matters: the model cannot memorise strategy, so predicting legal moves forces it to track the board.

In [ ]:
import time
from wmlab.othello.game import generate_games, replay, sq_name, TOK_TO_SQ
N_TRAIN = 20_000 if SMOKE else 500_000
path = f"{WORK}/othello_games_{N_TRAIN}.npy"
t = time.time()
if os.path.exists(path):
    games = np.load(path)
else:
    games = generate_games(N_TRAIN, seed=1, workers=os.cpu_count())
    np.save(path, games)
test_games = generate_games(1000 if SMOKE else 4000, seed=2, workers=os.cpu_count())
print(games.shape, f"{time.time()-t:.0f}s")
print("first game:", " ".join(sq_name(TOK_TO_SQ[t]) for t in games[0][:12]), "...")

## 2 · Train Othello-GPT
8 layers, 512 wide, about 25M parameters, as in Li et al. Expect 45–60 minutes on a T4. The number to watch is not the loss but the **legal-move rate**: the share of positions where the model's top guess is a legal move. A good run clears 99%.

In [ ]:
from wmlab.othello.model import OthelloGPT, GPTConfig, train, legal_move_rate
cfg = GPTConfig(n_layer=4, n_head=4, d_model=128) if SMOKE else GPTConfig(n_layer=8, n_head=8, d_model=512)
model = OthelloGPT(cfg)
ckpt = f"{WORK}/othello_gpt_{cfg.n_layer}x{cfg.d_model}_{N_TRAIN}.pt"
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.to(DEVICE)
    history = []
else:
    history = train(model, games, epochs=1 if SMOKE else 2, batch_size=128 if SMOKE else 256,
                    lr=1e-3 if SMOKE else 5e-4, device=DEVICE, log_every=500)
    torch.save(model.state_dict(), ckpt)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")
print("legal-move rate:", round(legal_move_rate(model, test_games, n=300, device=DEVICE), 4))
if history:
    plt.figure(figsize=(6, 3)); plt.plot(history, lw=.5); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Othello-GPT training"); plt.show()

In [ ]:
# Optional: rerun this cell until the legal-move rate clears ~0.98. An undertrained
# model still has a board inside it, but a fuzzier one: probes and edits both weaken.
history += train(model, games, epochs=1, batch_size=256, lr=2e-4, device=DEVICE, log_every=1000) if not SMOKE else []
torch.save(model.state_dict(), ckpt)
print("legal-move rate:", round(legal_move_rate(model, test_games, n=300, device=DEVICE), 4))

## 3 · Linear probes, layer by layer
Each square is labelled *empty*, *mine* or *theirs* relative to the player about to move. Li et al. first labelled squares black/white and needed a non-linear probe; Nanda found that the model represents the board relative to the mover, and in that basis a plain linear probe works. Worth sitting with: the model's native coordinates were not ours.

The **untrained-model** line is the control. A probe can learn some structure from any activations (the empty centre squares, for instance), so only the gap above that line is evidence of a world model.

In [ ]:
from wmlab.othello.probe import collect, train_probe, STATE_NAMES, relative_board, plot_board
n_probe = 800 if SMOKE else 3000
acts, labels = collect(model, test_games[:n_probe], device=DEVICE)
probes, accs = {}, []
for L in range(cfg.n_layer + 1):
    p, acc = train_probe(acts[L], labels, device=DEVICE)
    probes[L] = p; accs.append(acc.mean())
    print(f"layer {L}: probe accuracy {acc.mean():.3f}")

control = OthelloGPT(cfg).to(DEVICE)                      # same architecture, random weights
c_acts, _ = collect(control, test_games[:n_probe], device=DEVICE)
ctrl_accs = [train_probe(c_acts[L], labels, device=DEVICE)[1].mean() for L in range(cfg.n_layer + 1)]
del c_acts

plt.figure(figsize=(6, 3.2))
plt.plot(accs, "o-", label="trained Othello-GPT"); plt.plot(ctrl_accs, "s--", label="untrained control")
plt.xlabel("layer (0 = embeddings)"); plt.ylabel("probe accuracy, 64 squares"); plt.legend(); plt.grid(alpha=.3)
plt.title("Where does the board appear?"); plt.savefig(f"{WORK}/probe_by_layer.png", dpi=150, bbox_inches="tight"); plt.show()
BEST = int(np.argmax(accs)); print("best layer:", BEST)

### What the probe sees
One held-out position: the true board beside the board read out of the model's activations.

In [ ]:
g = test_games[0]; boards, nexts, legals = replay(g); t = 30
x = torch.from_numpy(g[None, :-1].astype(np.int64)).to(DEVICE)
with torch.no_grad():
    _, resids = model(x, return_resid=True)
    read = probes[BEST](resids[BEST][0, t].float()).view(64, 3).argmax(-1).cpu().numpy()
true = relative_board(boards[t], nexts[t])
fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
plot_board(ax[0], true, f"true board after move {t+1}")
plot_board(ax[1], read, f"probe, layer {BEST}: {(read == true).mean():.0%} squares right")
plt.show()

## 4 · Causal intervention
A probe shows the board is *decodable*. It does not show the model *uses* it. For that, change the board inside the model and see whether behaviour follows.

For each trial: pick a mid-game position, pick an occupied square whose colour flip would change the set of legal moves, push the activations along the probe's direction for that square, and compare the model's top-N predictions against the legal moves of the **original** and the **edited** board (N = number of legal moves; the score counts mistakes, so lower is better).

If the model uses its board, predictions should move toward the edited board's legal set.

In [ ]:
from wmlab.othello.probe import run_interventions
import itertools
results = {}
for start in range(max(1, BEST - 2), BEST + 1):
    layers = set(range(start, cfg.n_layer + 1))
    rows = run_interventions(model, probes, test_games, layers, n_trials=60 if SMOKE else 300, device=DEVICE)
    m = {k: np.mean([r[k] for r in rows]) for k in ("base_vs_old", "base_vs_new", "edit_vs_old", "edit_vs_new")}
    results[start] = (m, rows)
    print(f"edit layers {start}..{cfg.n_layer}:  before edit, errors vs new board {m['base_vs_new']:.2f}"
          f"  ->  after edit {m['edit_vs_new']:.2f}   (errors vs original board: {m['base_vs_old']:.2f} -> {m['edit_vs_old']:.2f})")

Read the line as: *after the edit, predictions got closer to the edited board and further from the original*. That crossover is the causal evidence. Now one example worth screenshotting.

In [ ]:
from wmlab.othello.probe import make_edit, flipped_board_legal, MINE, THEIRS
start = min(results, key=lambda s: results[s][0]["edit_vs_new"])
layers = set(range(start, cfg.n_layer + 1))
rng = np.random.default_rng(3)
for _ in range(500):
    g = test_games[rng.integers(len(test_games))]; boards, nexts, legals = replay(g)
    t = int(rng.integers(10, min(len(boards), cfg.ctx) - 1)); lab = relative_board(boards[t], nexts[t])
    sq = int(rng.choice(np.flatnonzero(lab != 0))); new_legal = flipped_board_legal(g, t, sq)
    if not new_legal or sorted(new_legal) == sorted(legals[t]): continue
    new_state = MINE if lab[sq] == THEIRS else THEIRS
    x = torch.from_numpy(g[None, :-1].astype(np.int64)).to(DEVICE)
    with torch.no_grad():
        base = model(x)[0, t]; edited = model(x, edit=make_edit(probes, t, sq, new_state, layers))[0, t]
    top = lambda lg, n: [TOK_TO_SQ.get(i, -1) for i in lg.topk(n).indices.tolist()]
    pb, pe = top(base, len(legals[t])), top(edited, len(new_legal))
    if set(pb) == set(legals[t]) and set(pe) == set(new_legal): break
lab2 = lab.copy(); lab2[sq] = new_state
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
plot_board(ax[0], lab, f"original · model's top moves (red)\nlegal: {sorted(map(sq_name, legals[t]))}", pb)
plot_board(ax[1], lab2, f"after flipping {sq_name(sq)} in activations\nlegal: {sorted(map(sq_name, new_legal))}", pe)
plt.tight_layout(); plt.savefig(f"{WORK}/intervention_example.png", dpi=150, bbox_inches="tight"); plt.show()

## 5 · Commit it

Copy `probe_by_layer.png` and `intervention_example.png` from your work folder into the repo's `results/week0/`, and write three sentences in `LOG.md`: what the probe found, what the control found, what the intervention showed. If something did not work, say so; the plan asks for failures too.

**Self-check before moving on:** explain, without notes, why a probe on an *untrained* network is the right control, and why decodability alone would not prove the model uses its board.